# 第 11 章 · 形状与广播：从右往左对齐，1 可以拉伸

本章回答：两个形状不一样的张量为什么能直接相加？什么时候会报 "size must match"？多头注意力里把 (B, T, D) 变成 (B, H, T, d) 的那几步变形，各自在干什么？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/11-shape-broadcast.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §4.2；官方文档 Broadcasting semantics（〔在线〕docs.pytorch.org/docs/stable/notes/broadcasting.html）；einops 文档（〔在线〕einops.rocks）；Transformer 讲义 [第 13 章 多头](../../../transformer-deep-dive/notes/lessons/13-multi-head.md)；本章输出在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
from einops import rearrange
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 广播规则 | 从右对齐 | `broadcast_shape(a, b)` | 四组：三组成功、(2,3,4)+(3,) 报错 |
| 第一步 加法表 | (3, 1) + (2,) | `addition_table()` | [[11, 21], [12, 22], [13, 23]] |
| 第二步 陷阱 | B = D 时不报错 | `keepdim_trap()` | (False, True) |
| 第三、四步 拆头 | view + transpose | `split_heads()` | (2, 2, 5, 4)、stride (40, 4, 8, 1)、与 einops 一致 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `addition_table`**（来自 `ch11_shape_broadcast.py`）：下面要用到的小工具。

In [ ]:
def addition_table():
    return torch.tensor([[1.0], [2.0], [3.0]]) + torch.tensor([10.0, 20.0])

**函数 `broadcast_shape`**（来自 `ch11_shape_broadcast.py`）：能广播返回结果形状，不能返回报错信息。

In [ ]:
def broadcast_shape(a: tuple, b: tuple):
    """能广播返回结果形状，不能返回报错信息。"""
    try:
        return tuple((torch.ones(a) + torch.ones(b)).shape)
    except RuntimeError as e:
        return str(e)

**函数 `keepdim_trap`**（来自 `ch11_shape_broadcast.py`）：B = D 时 x - x.mean(1) 不报错但错；keepdim 才对。返回两者是否一样。

In [ ]:
def keepdim_trap(n: int = 3):
    """B = D 时 x - x.mean(1) 不报错但错；keepdim 才对。返回两者是否一样。"""
    torch.manual_seed(0)
    x = torch.randn(n, n)
    wrong = x - x.mean(dim=1)
    right = x - x.mean(dim=1, keepdim=True)
    return torch.allclose(wrong, right), torch.allclose(right.mean(1), torch.zeros(n), atol=1e-6)

**函数 `split_heads`**（来自 `ch11_shape_broadcast.py`）：下面要用到的小工具。

In [ ]:
def split_heads(B=2, T=5, D=8, H=2):
    q = torch.randn(B, T, D)
    a = q.view(B, T, H, D // H).transpose(1, 2)
    b = rearrange(q, "b t (h d) -> b h t d", h=H)
    return tuple(a.shape), a.stride(), torch.equal(a, b)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 广播：")
for a, b in [((2, 3, 4), (4,)), ((2, 3, 4), (3, 1)), ((2, 3, 4), (3,)), ((3, 1), (2,))]:
    print(f"  {a} + {b} → {broadcast_shape(a, b)}")
print("加法表：\n", addition_table())

In [ ]:
print("\n第二步 keepdim 陷阱（错的等于对的？对的每行均值为 0？）：", keepdim_trap())

In [ ]:
print("\n第三、四步 拆头 (形状, stride, 与 einops 一致)：", split_heads())

## 3. 代码里才看得到的

1. **错误的广播在方阵上不报错**：`keepdim_trap` 故意用 3 × 3，错的结果照样算出来。
2. **拆头后 stride 是 (40, 4, 8, 1)**：第 1、2 维的 stride 不递减，就是不连续。

## 4. 自己改一改

- `broadcast_shape((5, 1, 4), (1, 3, 1))`，先猜再跑。
- 把 `keepdim_trap(4)` 改成非方阵（比如 3 × 4），看会不会报错。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 广播从右往左对，相等或者有一个 1；1 会拉伸不复制；
> 恰好对上最危险，规约记得 keepdim；
> 拆头先切最后维，再用 transpose 换位置；einops 读代码就是读形状。